# 🕵️ Jev "Can't Hallucinate". So We Tried to Make It Lie.

**Jev** is TypeSafe AI's new *decision* model: it doesn't write text, it only answers typed questions with probabilities.
Its makers make two big claims:

1. **It can't hallucinate.**
2. **Its answers are free.**

In this notebook we test both, using the messages scammers send us every day.

| Mark | Meaning |
|---|---|
| 🧑‍🏫 **Teach** | The Jev moments. Read every line (about 15 lines in total) |
| ▶️ **Just run** | Plumbing we prepared (`jev_helpers.py`). Run it and look at the result |

> ⚠️ A learning project, not a safety product. Never treat a model's answer as proof that a message is safe.

## ▶️ Setup

You need **one** key from [OpenRouter](https://openrouter.ai/settings/keys), with a few dollars of credit (this notebook uses well under $1).
Add it in Colab's 🔑 **Secrets** panel as `OPENROUTER_API_KEY` and switch on notebook access.

In [ ]:
!pip install -q typesafe-sdk openai gradio pandas matplotlib
!wget -q -nc https://raw.githubusercontent.com/Kiran-B23/jev-scam-detector/main/jev_helpers.py
!mkdir -p data && wget -q -nc -P data https://raw.githubusercontent.com/Kiran-B23/jev-scam-detector/main/data/scam_messages.csv

from jev_helpers import *
KEY = get_api_key()

---
# Claim #1: "It can't hallucinate". Let's try to make it lie.

## 🧑‍🏫 First, the old way: asking a chatbot

The usual way to check a message: ask a chatbot, ask for JSON, and hope.

In [ ]:
sms = ("Dear consumer, your electricity power will be disconnected tonight at 9:30 PM "
       "because previous month bill was not updated. Call officer 9XXXX X7788 immediately.")

reply = ask_llm('Is this SMS a scam? Reply in JSON like {"is_scam": true}.\n\n' + sms)
print(repr(reply))

import json
try:
    print("Parsed:", json.loads(reply))
except json.JSONDecodeError as e:
    print("💥 json.loads crashed:", e)

## 🧑‍🏫 The swap: two lines

Same SMS, same question, but asked to Jev. We give it the **state** (the thing to judge) and a typed **question**.

In [ ]:
from typesafe_sdk import TypeSafeClient, Noul, Choice, Score
jev = TypeSafeClient(api_key=KEY, base_url="https://openrouter.ai/api", model="jev-1.13")

answer = jev.system_one(state=sms, questions={"is_scam": Noul(instructions="Is this SMS a scam?")})
answer.nouls["is_scam"].noul

A **number**, the probability that the answer is *yes*. No JSON, nothing to parse, and it can't come back in the wrong shape.
That's what "can't hallucinate" really means: **the format is guaranteed**. Whether the *answer* is right is what we test next.

## ▶️ Ask more, in the same call

Jev has three question types: **Noul** (yes/no), **Choice** (pick one), **Score** (how much). We'll ask one of each.
The options are just plain-English descriptions.

In [ ]:
SCAM_TYPES = {
    "fake_refund_or_upi": "Fake refund, cashback or 'scan QR / enter PIN to receive money'",
    "kyc_or_account_block": "KYC, PAN, SIM or bank account 'will be blocked'",
    "job_or_task": "Part-time job or task that needs a fee or deposit",
    "prize_or_lottery": "Lottery, lucky draw, prize or reward points",
    "police_or_parcel_threat": "Courier, customs, police or 'digital arrest' threats",
    "bill_disconnection": "Electricity, gas, broadband or mobile 'will be cut' today",
    "fake_family_or_boss": "Pretends to be family, a friend or a boss and asks for money or codes",
    "investment_or_loan": "Guaranteed returns, stock tips, crypto or instant loans",
    "not_scam": "A normal, genuine message",
}

QUESTIONS = {
    "is_scam": Noul(
        instructions="Is this message a scam or fraud attempt?",
        criteria={"true": "Tries to trick the reader into paying, sharing a code, clicking or calling",
                  "false": "A genuine message, even if it mentions OTPs, money or deadlines"},
    ),
    "scam_type": Choice(instructions="What kind of message is this?", criteria=SCAM_TYPES),
    "pressure": Score(instructions="How much pressure does it put on the reader?",
                      criteria=["None", "Mild deadline", "Strong urgency", "Threats"]),
}

a = jev.system_one(state=sms, questions=QUESTIONS)
print("Scam:", a.nouls["is_scam"].noul, "| Type:", a.choices["scam_type"].choice, "| Pressure:", a.scores["pressure"].score, "/ 3")

## 🧑‍🏫 Try to make it lie

Five messages built to fool a quick glance. **Before you run it, guess each one: scam or genuine?**

In [ ]:
TRICKY = [
    "Hello, I am Priya from a recruitment agency. We saw your profile. Simple task: rate hotels online, Rs 150 per task. Interested?",
    "123456 is your OTP for login. Do not share it with anyone. Bank staff will never ask for your OTP.",
    "Your courier is on hold due to incomplete address. Pay Rs 25 redelivery fee: parcel-redeliver.example",
    "Bhai galti se aapke account me 5000 transfer ho gaya, please wapas bhej do. Screenshot bhej raha hoon.",
    "Hi! Your cab driver Ramesh is arriving in 3 mins. OTP to start the ride: 4821.",
]

for msg in TRICKY:
    p = jev.system_one(state=msg, questions=QUESTIONS).nouls["is_scam"].noul
    print(f"{p:.2f}  {msg[:70]}...")

**The answers:** scam · genuine · scam · scam · genuine. Did Jev fool you, or did you fool Jev?

---
# Claim #2: "Its answers are free". Are they?

## ▶️ The race: Jev vs a chatbot on 100 messages

100 made-up messages (60 scams, 40 genuine, about a third deliberately tricky). Same questions, same rules: 10 at a time each.

In [ ]:
df = load_messages()
results = await race(df, QUESTIONS, SCAM_TYPES)

In [ ]:
scoreboard(results)

---
# We caught it lying

## ▶️ Wrong, and sure of it

Jev can't give a *broken* answer. But can it give a **wrong** one, and be confident about it?

In [ ]:
confident_mistakes(results)

---
# The fix (it's 3 lines)

## 🧑‍🏫 Confidence bands, with a slider

Jev tells us how sure it is, so **we** decide when to trust it. Drag the slider, re-run, and watch the bands change.

In [ ]:
threshold = 0.9  #@param {type:"slider", min:0.5, max:0.99, step:0.01}

def verdict(p):
    if p >= threshold:      return "🚨 Scam"
    if p <= 1 - threshold:  return "✅ Looks normal"
    return "⚠️ Not sure: get a second opinion"

band_report(results, verdict)

Stricter slider → fewer mistakes, but more messages land in ⚠️. For those, we ask the chatbot for a second opinion.
Jev decides *every* message fast and cheap, and the slow chatbot only sees the tricky few.

---
## ▶️ Bonus: turn it into an app you can share

In [ ]:
check = make_checker(jev, QUESTIONS, verdict)
launch_app(check, examples=[[sms], [TRICKY[0]], [TRICKY[4]]])

---
## The verdict
- **Claim #1, "can't hallucinate":** ✅ the *format* is guaranteed, but ⚠️ it can still be **wrong, and confident**. That's why the slider matters.
- **Claim #2, "free answers":** ✅ output is free, and you only pay for what you send. See the race for the real bill.
- **Worth the hype?** For fast, high-volume yes/no and pick-one decisions, with a threshold and a fallback, yes. As a replacement for chatbots, no.

**Your turn:** add a scam message you've received (remove personal details first) to `TRICKY` and see if it fools Jev.